# Discovery + Silver: `crm.leads`

Prospectos comerciales sueltos. Sin FK -- son independientes de `accounts`/`contacts` (todavia no se convirtieron en cliente).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.crm__leads", engine)
df.shape

(2000, 11)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

lead_id                 object
first_name              object
last_name               object
email                   object
source                  object
status                  object
score                   object
created_at              object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,lead_id,first_name,last_name,email,source,status,score,created_at,_source_file,_ingested_at,_dag_run_id
0,LED-0000001,Javier,Torres,javier.torres9271@synthetic.dev,web,qualified,49,2022-01-13 15:20:06,crm/leads.csv,2026-07-15 22:29:48.108464,manual__2026-07-15T22:29:45+00:00
1,LED-0000002,Eduardo,Arancibia,eduardo.arancibia7632@demo.io,cold_call,contacted,61,2024-08-21 15:32:50,crm/leads.csv,2026-07-15 22:29:48.108464,manual__2026-07-15T22:29:45+00:00
2,LED-0000003,Fernanda,Vasquez,fernanda.vasquez1846@example.com,referral,qualified,43,2025-10-28 22:38:01,crm/leads.csv,2026-07-15 22:29:48.108464,manual__2026-07-15T22:29:45+00:00
3,LED-0000004,Agustina,Diaz,agustina.diaz8925@mail.test,web,qualified,0,2023-07-17 22:09:48,crm/leads.csv,2026-07-15 22:29:48.108464,manual__2026-07-15T22:29:45+00:00
4,LED-0000005,Camila,Riquelme,camila.riquelme2816@lake.local,web,new,7,2022-07-19 08:52:51,crm/leads.csv,2026-07-15 22:29:48.108464,manual__2026-07-15T22:29:45+00:00


## 2. Nulos y duplicados

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("lead_id duplicados:", df["lead_id"].duplicated().sum())

Nulos por columna:
lead_id         0
first_name      0
last_name       0
email           0
source          0
status          0
score           0
created_at      0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

lead_id duplicados: 0


## 3. `source`, `status`, `score`

In [4]:
print("source:")
print(df["source"].value_counts())
print()
print("status:")
print(df["status"].value_counts())
print()
score = pd.to_numeric(df["score"], errors="coerce")
print("score fuera de [0, 100]:", ((score < 0) | (score > 100)).sum())

source:
source
web          810
referral     402
event        302
ads          282
cold_call    204
Name: count, dtype: int64

status:
status
new          594
contacted    525
qualified    395
lost         281
converted    205
Name: count, dtype: int64

score fuera de [0, 100]: 0


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, `score` en rango). Solo tipado y estandarizacion.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["lead_id", "first_name", "last_name", "email", "source", "status", "score", "created_at"]].copy()

df_silver["first_name"] = df_silver["first_name"].str.strip()
df_silver["last_name"] = df_silver["last_name"].str.strip()
df_silver["email"] = df_silver["email"].str.strip().str.lower()
df_silver["source"] = df_silver["source"].str.strip().str.lower()
df_silver["status"] = df_silver["status"].str.strip().str.lower()
df_silver["score"] = pd.to_numeric(df_silver["score"], errors="raise").astype(int)
df_silver["created_at"] = pd.to_datetime(df_silver["created_at"])

df_silver.head()

,lead_id,first_name,last_name,email,source,status,score,created_at
0,LED-0000001,Javier,Torres,javier.torres9271@synthetic.dev,web,qualified,49,2022-01-13 15:20:06
1,LED-0000002,Eduardo,Arancibia,eduardo.arancibia7632@demo.io,cold_call,contacted,61,2024-08-21 15:32:50
2,LED-0000003,Fernanda,Vasquez,fernanda.vasquez1846@example.com,referral,qualified,43,2025-10-28 22:38:01
3,LED-0000004,Agustina,Diaz,agustina.diaz8925@mail.test,web,qualified,0,2023-07-17 22:09:48
4,LED-0000005,Camila,Riquelme,camila.riquelme2816@lake.local,web,new,7,2022-07-19 08:52:51


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["lead_id"].is_unique
assert df_silver["score"].between(0, 100).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 2000 filas listas para silver


## 7. Escribir en `silver.crm__leads`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "crm__leads",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)
print("Escrito en silver.crm__leads")

Escrito en silver.crm__leads


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.crm__leads LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT lead_id) AS ids_unicos FROM silver.crm__leads", engine))
check

   filas  ids_unicos
0   2000        2000


,lead_id,first_name,last_name,email,source,status,score,created_at,_silver_loaded_at
0,LED-0000001,Javier,Torres,javier.torres9271@synthetic.dev,web,qualified,49,2022-01-13 15:20:06,2026-07-16 19:37:34.204045+00:00
1,LED-0000002,Eduardo,Arancibia,eduardo.arancibia7632@demo.io,cold_call,contacted,61,2024-08-21 15:32:50,2026-07-16 19:37:34.204045+00:00
2,LED-0000003,Fernanda,Vasquez,fernanda.vasquez1846@example.com,referral,qualified,43,2025-10-28 22:38:01,2026-07-16 19:37:34.204045+00:00
3,LED-0000004,Agustina,Diaz,agustina.diaz8925@mail.test,web,qualified,0,2023-07-17 22:09:48,2026-07-16 19:37:34.204045+00:00
4,LED-0000005,Camila,Riquelme,camila.riquelme2816@lake.local,web,new,7,2022-07-19 08:52:51,2026-07-16 19:37:34.204045+00:00
